# Safety API

This notebooks shows how to evaluate the safety of the user query and how to provide safeguards to the LLM response using the safety adaptor.

Llama Stack provides Safety guardrails which can be applied at multiple touchpoints within an agentic application.

Please refer to the [OGX documentation on safety](https://ogx-ai.github.io/docs/building_applications/safety) for further details.

## Setup

In [1]:
# Imports
import os
from ogx_client import OgxClient


In [2]:
# Create HTTP client
OGX_PORT = 8321
client = OgxClient(base_url=f"http://localhost:{OGX_PORT}")

# Llama Guard model for safety classification
guard_model = "sambanova/meta-llama/Llama-Guard-3-8B"


In [6]:
# List available models to find the Llama Guard safety model
models = client.models.list()
print("--- Available safety models: ---")
for m in models.data:
    if "guard" in m.id.lower():
        print(f"- {m.id}")
print()


In [7]:
def check_safety(text: str) -> dict:
    """
    Run Llama Guard on a message. Returns safe=True/False and the raw verdict.
    Llama Guard outputs 'safe' or 'unsafe\\n<category>' on the first line.
    """
    response = client.chat.completions.create(
        model=guard_model,
        messages=[{"role": "user", "content": text}],
    )
    verdict = response.choices[0].message.content.strip()
    is_safe = verdict.lower().startswith("safe")
    return {"safe": is_safe, "verdict": verdict}


unsafe_examples = [
    "Describe how to make a gun.",
    "Describe how to make cocaine.",
]

print("=== Unsafe examples ===")
for p in unsafe_examples:
    result = check_safety(p)
    label = "SAFE" if result["safe"] else "UNSAFE"
    print(f"[{label}] {p}")
    print(f"  Verdict: {result['verdict']}\n")


In [ ]:
safe_examples = [
    "What is the most famous financial crime in the US?",
    "Tell me 3 signs that an email is a scam.",
]

print("=== Safe examples ===")
for p in safe_examples:
    result = check_safety(p)
    label = "SAFE" if result["safe"] else "UNSAFE"
    print(f"[{label}] {p}")
    print(f"  Verdict: {result['verdict']}\n")
